# Initiation à la recherche M2 ICE
Ce projet a pour but de vous faire créer **une simulation réaliste du travail de recherche** : vous allez constituer une équipe, définir un sujet, choisir une conférence ou revue adaptée, construire un **état de l’art réel**, puis rédiger une **introduction et un résumé d’article scientifique** à partir de ces sources.

Toutes les étapes reposent sur des requêtes à un **LLM réel**, interrogé grâce à la petite bibliothèque Python `llm` fournie avec ce notebook, afin d’obtenir des données **fiables et exploitables en Python**.
Chaque résultat devra être renvoyé sous un **format structuré (JSON, objet)** afin d’être réutilisé dans les étapes suivantes.

Dans un premier temps, vous enchaînerez vous-même les étapes, une par une. À la fin du projet, vous construirez un **agent** : c'est le LLM qui décidera d'appeler chacune de vos briques comme un outil (*tool*) pour produire et sauvegarder l'article complet, sans intervention de votre part.

> 📚 Les encadrés comme celui-ci décrivent le fonctionnement de la bibliothèque `llm`. Ils apparaissent **au fur et à mesure**, quand vous en avez besoin.

### Installation des packages
Le dossier `llm/` doit se trouver **dans le même dossier que ce notebook**.

In [6]:
pip install openai pydantic requests pypdf markdown-pdf

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import io
import json
import requests
from pathlib import Path
from pydantic import BaseModel, Field
from typing import Optional

from llm import OpenModel, ModelMessage, Prompt, Tool

### Clé d'API et modèle
La clé d'API et le nom du modèle à utiliser vous sont **fournis pour le TP** : recopiez-les dans la cellule ci-dessous.

- Ne partagez pas cette clé et ne la déposez pas sur un dépôt public (pensez à l'effacer avant de rendre votre notebook).
- La clé est commune à tout le groupe et son budget est limité : évitez de relancer inutilement des requêtes, en particulier dans des boucles.

In [8]:
API_KEY = ""      # clé d'API fournie pour le TP (retirée avant dépôt Git / rendu)
MODEL_NAME = "deepseek/deepseek-v4.1-flash-1017"   # nom du modèle fourni pour le TP

# Utilisation d'un LLM avec la bibliothèque `llm`

> 📚 **Bibliothèque `llm` — se connecter à un modèle**
>
> La bibliothèque passe par [OpenRouter](https://openrouter.ai), un service qui donne accès à de nombreux LLM avec une seule clé. Tout part d'un objet `OpenModel`, qui représente **un modèle et ses réglages** :
>
> ```python
> modele = OpenModel(MODEL_NAME, api_key=API_KEY, temperature=0.7, max_output_tokens=1200)
> ```
>
> - `model_name` : le nom du modèle ;
> - `api_key` : la clé d'API, passée en paramètre ;
> - `temperature` : 0 = réponses très stables, 1 et plus = réponses plus variées ;
> - `max_output_tokens` : longueur maximale de la réponse (à augmenter pour les textes longs).
>
> Ces réglages sont fixés **à la création** de l'objet : pour utiliser une autre température, créez un second `OpenModel`.
> En cas d'erreur de l'API, la requête est automatiquement relancée (`max_retries=4` par défaut).

## Exemples d'utilisation

### Requête simple

> 📚 **Bibliothèque `llm` — générer du texte**
>
> `modele.generate_text_from_prompt(prompt)` envoie le prompt au modèle et renvoie sa réponse sous forme de **chaîne de caractères**.

In [9]:
modele = OpenModel(MODEL_NAME, api_key=API_KEY, temperature=1.)

ValueError: An OpenRouter API key is required (api_key or OPENROUTER_API_KEY).

In [ ]:
prompt = "Invente le titre d'un article en français sur l'intelligence artificielle et les sciences humaines. Une seule proposition"

# un second modèle, plus "créatif", pour cette requête
modele_creatif = OpenModel(MODEL_NAME, api_key=API_KEY, temperature=.7)
print(modele_creatif.generate_text_from_prompt(prompt))

« L’IA face aux humanités : quand les algorithmes réapprennent à penser l’humain »

*(Autre proposition possible : « Humanités augmentées : ce que l’intelligence artificielle change à la lecture des textes, des héritages et des savoirs »)*


### Requête Json
Ce mode est pratique mais il est difficile de récupérer l'information que l'on veut. Il est possible de demander au LLM de répondre sous forme de json (c'est la base des agents LLM). Voici un exemple :

> 📚 **Bibliothèque `llm` — générer du JSON**
>
> `modele.generate_json_from_prompt(prompt)` force le modèle à répondre en JSON et renvoie directement un **dictionnaire Python** (le `json.loads` est fait pour vous). Si le JSON reçu est mal formé, la requête est relancée.
> Le prompt doit décrire les clés attendues.

In [ ]:
prompt = f"""
Tu es un assistant qui renvoie toujours un JSON **valide**.
Invente le titre d'un article en français sur l'intelligence artificielle et les sciences humaines.
Réponds UNIQUEMENT en JSON, sans texte autour.
Valeurs possibles :
    - "titre" : titre de l'article
    - "key" : liste de 5 keywords
"""

resp = modele.generate_json_from_prompt(prompt)
print("titre :", resp['titre'])
print("keywords :", resp['key'])

titre : L'intelligence artificielle à l'épreuve des sciences humaines : repenser l'humain à l'ère des algorithmes
keywords : ['intelligence artificielle', 'sciences humaines', 'éthique', 'interdisciplinarité', 'humanités numériques']


### Requête Objet
Il est possible de structurer encore mieux les réponses d'un llm en utilisant le formalisme objet

> 📚 **Bibliothèque `llm` — générer un objet**
>
> On décrit la réponse attendue par une classe [Pydantic](https://docs.pydantic.dev) (`BaseModel`), puis on la passe dans `text_format` :
>
> ```python
> objet = modele.generate_object_from_prompt(prompt, text_format=MaClasse)
> ```
>
> Le résultat est une **instance de la classe**, déjà validée (types vérifiés) : on accède aux champs avec `objet.champ`, et `objet.model_dump()` la convertit en dictionnaire.
> Une classe peut contenir des listes (`list[str]`) ou d'autres classes Pydantic, et `Field(description="...")` permet d'expliquer un champ au modèle.

> 📚 **Bibliothèque `llm` — construire une conversation avec `ModelMessage`**
>
> Les méthodes `..._from_prompt` envoient un seul message. Pour séparer les consignes générales (message **system**) de la demande (message **user**), on construit la conversation avec `ModelMessage` :
>
> ```python
> messages = ModelMessage()
> messages.add_system_message("consignes générales, rôle du modèle")
> messages.add_user_message("la demande")
> ```
>
> puis on passe `messages.get_inputs()` à la version « conversation » de chaque méthode :
>
> | un seul prompt | conversation | résultat |
> |---|---|---|
> | `generate_text_from_prompt(prompt)` | `generate_text(inputs)` | `str` |
> | `generate_json_from_prompt(prompt)` | `generate_json(inputs)` | `dict` |
> | `generate_object_from_prompt(prompt, text_format=C)` | `generate_object(inputs, text_format=C)` | instance de `C` |

In [ ]:
class Titre(BaseModel):
    titre: str
    keywords: list[str]

    def affiche(self):
        print("titre :", self.titre)
        print(f"""keywords : {", ".join(self.keywords)}""")

NameError: name 'BaseModel' is not defined

In [ ]:
prompt_titre = f"""
Tu es un générateur de titre d'article **réalistes** en France.
Crée un titre d'article sous la forme suivante  :
- `titre` : crédible en fonction des thématiques demandées
- `keywords` : une liste de mots associés, au moins 4
⚠️ La sortie **doit** respecter le schéma JSON fourni (aucun texte hors JSON).
"""
prompt_user = "Invente le titre d'un article en français sur l'intelligence artificielle et les sciences humaines."

messages = ModelMessage()
messages.add_system_message(prompt_titre)
messages.add_user_message(prompt_user)

title = modele.generate_object(messages.get_inputs(), text_format=Titre)
title.affiche()

titre : Intelligence artificielle et sciences humaines : vers une nouvelle alliance des savoirs
keywords : intelligence artificielle, sciences humaines, éthique, interdisciplinarité, humanités numériques, recherche


### Prompts dans des fichiers Markdown

Dans les exemples précédents, le prompt est écrit directement dans le code. C'est acceptable pour deux lignes, mais un vrai prompt est long : mélangé au code, il rend le notebook illisible. On range donc **chaque prompt dans son propre fichier Markdown**, dans le dossier `prompts/`, et le notebook ne contient plus que le code.

> 📚 **Bibliothèque `llm` — les gabarits de prompt avec `Prompt`**
>
> La classe `Prompt` charge un prompt depuis un fichier Markdown. Le fichier peut contenir des **variables** notées `{{nom}}`, ce qui permet de réutiliser le même prompt avec des valeurs différentes (sujet, conférence visée…) :
>
> - `Prompt(Path("prompts/mon_prompt.md"))` charge le fichier ;
> - `gabarit.set("sujet", "...")` donne une valeur à une variable, `gabarit.assign({...})` à plusieurs ;
> - `gabarit.render()` renvoie le prompt final, prêt à être envoyé au modèle ;
> - `gabarit.get_wildcards()` renvoie le dictionnaire des variables et de leurs valeurs.
>
> Les valeurs doivent être des chaînes de caractères. Une variable sans valeur est remplacée par une chaîne vide : affichez `gabarit.render()` pour vérifier ce qui est réellement envoyé.
> Pour un essai rapide, `Prompt(text="... {{sujet}} ...")` crée un gabarit sans fichier.

Le fichier `prompts/exemple_titre.md` contient :

```markdown
Invente le titre d'un article en {{langue}} sur {{sujet}}.

Une seule proposition, sans commentaire.
```

In [ ]:
gabarit_titre = Prompt(Path("prompts/exemple_titre.md"))
print("Variables :", gabarit_titre.get_wildcards())

gabarit_titre.assign({
    "langue": "français",
    "sujet": "l'intelligence artificielle et les sciences humaines",
})
print("Prompt envoyé :", gabarit_titre.render())
print(modele.generate_text_from_prompt(gabarit_titre.render()))

# le même gabarit, avec un autre sujet
gabarit_titre.set("sujet", "l'éthique de la recherche")
print(modele.generate_text_from_prompt(gabarit_titre.render()))

Variables : {'langue': '', 'sujet': ''}
Prompt envoyé : Invente le titre d'un article en français sur l'intelligence artificielle et les sciences humaines.

Une seule proposition, sans commentaire.



« Quand les algorithmes lisent Hannah Arendt : l’IA à l’épreuve des sciences humaines »


« Éthique de la recherche : quand la quête de savoir doit composer avec la dignité humaine »


### Lecture d'un pdf sur le web
Le modèle ne va pas chercher lui-même un document sur le web : on télécharge le pdf, on en extrait le texte en Python, puis on place ce texte dans la conversation.

In [ ]:
from pypdf import PdfReader

import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)   # masque les avertissements de lecture des pdf

def lire_pdf(url, max_caracteres=20000):
    """Télécharge un pdf et renvoie son texte, limité à max_caracteres caractères."""
    reponse = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
    reponse.raise_for_status()
    lecteur = PdfReader(io.BytesIO(reponse.content))
    texte = "\n".join(page.extract_text() or "" for page in lecteur.pages)
    return texte[:max_caracteres]

texte_article = lire_pdf("https://arxiv.org/pdf/1706.03762")
print(texte_article[:300])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


In [ ]:
# les consignes sont dans prompts/exemple_analyse_article.md
prompt_analyse = Prompt(Path("prompts/exemple_analyse_article.md"))

messages = ModelMessage()
messages.add_system_message(prompt_analyse.render())
messages.add_user_message(texte_article)

resp = modele.generate_json(messages.get_inputs())
print(resp)

{'titre': 'Attention Is All You Need', 'key': ['Transformer', 'self-attention', 'machine translation', 'sequence transduction', 'encoder-decoder'], 'author': ['Ashish Vaswani', 'Noam Shazeer', 'Niki Parmar', 'Jakob Uszkoreit', 'Llion Jones', 'Aidan N. Gomez', 'Łukasz Kaiser', 'Illia Polosukhin'], 'summary': "Les auteurs proposent le Transformer, une nouvelle architecture de réseau de neurones basée uniquement sur des mécanismes d'attention, sans récurrence ni convolution, qui obtient des résultats de pointe en traduction automatique tout en étant plus parallélisable et plus rapide à entraîner."}


# Projet

**Règle pour tout le projet : un prompt = un fichier Markdown.** Aucun prompt ne doit être écrit dans le notebook. Chaque prompt (consignes système, demandes, descriptions d'outils) est rangé dans son propre fichier du dossier `prompts/`, avec un nom explicite (`prompts/financements.md`, `prompts/resume_article.md`…) et des variables `{{...}}` pour tout ce qui change d'un appel à l'autre. Le notebook ne contient que le code qui charge ces fichiers avec `Prompt`. Le dossier `prompts/` fait partie du rendu.

**Conseil pour tout le projet :** écrivez chaque étape sous la forme d'une **fonction Python** qui prend des paramètres simples (le sujet, des mots-clés, une adresse de pdf…) et renvoie un résultat structuré (objet ou dictionnaire). À la fin du projet (partie 7), ces fonctions deviendront les outils de votre agent : plus elles seront propres et indépendantes, plus cette dernière partie sera facile.

## 1. Création de l’équipe de recherche

Constituer une équipe fictive mais crédible, intégrée à un environnement de recherche réaliste (universités, CNRS, INRIA, etc.).
1. **Recherche des types de financements**
   - Utilisez le modèle pour rechercher et lister les principaux types de financement d’un doctorat en France.
   - Le résultat doit préciser : nom du dispositif, organisme, durée, contraintes.
2. **Création de votre profil de doctorant**
   - Décrivez vos intérêts de recherche, votre parcours universitaire, et votre motivation scientifique.
   - Utilisez le modèle pour générer un profil synthétique (format JSON) : `{"nom":..., "formation":..., "financement":..., "sujet":...}`.
3. **Recherche des types de chercheurs, de laboratoires et d’instituts**
   - Demandez au modèle de lister les rôles (Maître de Conférences, Chargé de Recherche, Ingénieur de Recherche, etc.), les types de laboratoires (UMR, équipes-projets INRIA, etc.), et les tutelles typiques.
   - Utilisez le modèle pour créer deux encadrants fictifs cohérents avec votre sujet.
   - Chaque encadrant doit avoir un **rôle, une spécialité, une affiliation et un champ de recherche précis**.
   - Structure attendue :
     `{"nom":..., "rôle":..., "institution":..., "axe_recherche":...}`

> 📚 **Bibliothèque `llm` — obtenir une liste d'objets**
>
> `generate_object` renvoie **un** objet. Pour obtenir une liste (de financements, d'encadrants…), créez une classe « enveloppe » qui contient la liste :
>
> ```python
> class Financement(BaseModel):
>     nom: str
>     organisme: str
>
> class ListeFinancements(BaseModel):
>     financements: list[Financement]
> ```
>
> et passez `text_format=ListeFinancements`.

In [ ]:
from typing import Literal
import urllib.parse
import xml.etree.ElementTree as ET
from pydantic import model_validator

class ObjetStruct(BaseModel):
    """Classe de base Pydantic tolérante au format de sérialisation imbriqué d'OpenRouter."""
    @model_validator(mode="before")
    @classmethod
    def _normaliser_openrouter(cls, v):
        if isinstance(v, str):
            try:
                v = json.loads(v)
            except Exception:
                return v
        if isinstance(v, dict) and "fields" in v and isinstance(v.get("fields"), list):
            res = {}
            for k, val in v["fields"]:
                if isinstance(val, dict) and "value" in val:
                    val = val["value"]
                elif isinstance(val, dict) and "elements" in val:
                    val = [x.get("value", x) if isinstance(x, dict) else x for x in val["elements"]]
                res[k] = val
            return res
        return v

# Thématique et informations de base pour notre projet de recherche
SUJET_RECHERCHE = (
    "Compression et distillation de grands modèles de langage (Small Language Models) "
    "pour l'exécution locale à faible consommation énergétique"
)
MOTS_CLES_INITIAUX = "knowledge distillation, small language models, quantization-aware training, edge AI, model compression"
DOMAINE_RECHERCHE = "Intelligence Artificielle, Traitement Automatique des Langues (NLP) et Green AI"
FORMATION_CANDIDAT = "Master 2 Ingénierie Computationnelle et Intelligence Artificielle (M2 ICE)"
INTERETS_CANDIDAT = (
    "Modèles de langage compacts (SLM de 1 à 3B paramètres), distillation de connaissances, "
    "quantification basse précision (INT4/INT8), inférence embarquée et frugalité énergétique"
)
MOTIVATION_CANDIDAT = (
    "Démocratiser l'accès aux modèles de langage tout en réduisant drastiquement leur empreinte "
    "énergétique et matérielle grâce à l'exécution locale sur des terminaux grand public sans GPU serveur."
)

# --- 1.1 Recherche des types de financements ---
class Financement(ObjetStruct):
    nom: str = Field(description="Nom du dispositif de financement doctoral")
    organisme: str = Field(description="Organisme financeur ou gestionnaire")
    duree: str = Field(description="Durée standard du financement")
    contraintes: str = Field(description="Contraintes, éligibilité ou contreparties")

class ListeFinancements(ObjetStruct):
    financements: list[Financement]

def rechercher_financements(domaine: str = DOMAINE_RECHERCHE) -> ListeFinancements:
    """Recherche et liste les principaux dispositifs de financement d'un doctorat en France."""
    gabarit = Prompt(Path("prompts/financements.md"))
    gabarit.set("domaine", domaine)
    return modele.generate_object_from_prompt(gabarit.render(), text_format=ListeFinancements)

# --- 1.2 Création du profil de doctorant (format JSON) ---
def creer_profil_doctorant(
    sujet: str = SUJET_RECHERCHE,
    formation: str = FORMATION_CANDIDAT,
    interets: str = INTERETS_CANDIDAT,
    motivation: str = MOTIVATION_CANDIDAT,
    financements_dispo: Optional[ListeFinancements] = None,
) -> dict:
    """Génère un profil synthétique de doctorant au format JSON."""
    if financements_dispo is None:
        texte_financements = "Contrat doctoral MESR, Convention CIFRE (ANRT), Projet ANR, Bourse Inria, Financement ADEME"
    else:
        texte_financements = ", ".join(f"{f.nom} ({f.organisme})" for f in financements_dispo.financements)

    gabarit = Prompt(Path("prompts/profil_doctorant.md"))
    gabarit.assign({
        "formation": formation,
        "interets": interets,
        "motivation": motivation,
        "sujet": sujet,
        "financements": texte_financements,
    })
    return modele.generate_json_from_prompt(gabarit.render())

# --- 1.3 Recherche des types de chercheurs/laboratoires et création des encadrants ---
class RoleChercheur(ObjetStruct):
    intitule: str = Field(description="Titre du poste (ex. Maître de Conférences, Chargé de Recherche)")
    statut: str = Field(description="Fonction publique ou statut associé et missions principales")
    peut_diriger_these_hdr: bool = Field(description="Vrai si le poste inclut habituellement l'HDR")

class TypeLaboratoire(ObjetStruct):
    sigle_ou_nom: str = Field(description="Type de structure (ex. UMR, Équipe-projet Inria, UAR)")
    description: str = Field(description="Fonctionnement et organisation")

class EcosystemeRecherche(ObjetStruct):
    roles_chercheurs: list[RoleChercheur]
    types_laboratoires: list[TypeLaboratoire]
    tutelles_typiques: list[str]

def rechercher_ecosysteme(domaine: str = DOMAINE_RECHERCHE) -> EcosystemeRecherche:
    """Liste les rôles des chercheurs, types de laboratoires et tutelles en France."""
    gabarit = Prompt(Path("prompts/ecosysteme_recherche.md"))
    gabarit.set("domaine", domaine)
    return modele.generate_object_from_prompt(gabarit.render(), text_format=EcosystemeRecherche)

class Encadrant(ObjetStruct):
    nom: str = Field(description="Prénom et nom de l'encadrant")
    role: str = Field(description="Rôle académique (ex. Directeur de Recherche Inria, Maîtresse de Conférences)")
    specialite: str = Field(description="Spécialité scientifique précise")
    institution: str = Field(description="Laboratoire (UMR / équipe-projet) et tutelles")
    axe_recherche: str = Field(description="Champ de recherche précis en lien avec le sujet")

    def format_attendu(self) -> dict:
        return {
            "nom": self.nom,
            "rôle": f"{self.role} (Spécialité : {self.specialite})",
            "institution": self.institution,
            "axe_recherche": self.axe_recherche,
        }

class EquipeEncadrants(ObjetStruct):
    encadrants: list[Encadrant]

def creer_encadrants(
    sujet: str = SUJET_RECHERCHE,
    profil_doctorant: Optional[dict] = None,
    ecosysteme: Optional[EcosystemeRecherche] = None,
) -> EquipeEncadrants:
    """Génère deux encadrants fictifs cohérents avec le sujet de recherche."""
    gabarit = Prompt(Path("prompts/encadrants.md"))
    gabarit.assign({
        "sujet": sujet,
        "profil_doctorant": json.dumps(profil_doctorant or {"sujet": sujet}, ensure_ascii=False),
        "ecosysteme": json.dumps(ecosysteme.model_dump() if ecosysteme else {}, ensure_ascii=False),
    })
    return modele.generate_object_from_prompt(gabarit.render(), text_format=EquipeEncadrants)

# --- Exécution de la Partie 1 ---
print("=== 1.1 Principaux financements doctoraux en France ===")
liste_financements = rechercher_financements()
for fin in liste_financements.financements:
    print(f"- {fin.nom} ({fin.organisme}, {fin.duree}) : {fin.contraintes}")

print("\n=== 1.2 Profil synthétique du doctorant (JSON) ===")
profil_doc = creer_profil_doctorant(financements_dispo=liste_financements)
print(json.dumps(profil_doc, indent=2, ensure_ascii=False))

print("\n=== 1.3 Écosystème de recherche et création des 2 encadrants ===")
ecosysteme_fr = rechercher_ecosysteme()
print("Rôles identifiés :", ", ".join(r.intitule for r in ecosysteme_fr.roles_chercheurs))
print("Structures :", ", ".join(l.sigle_ou_nom for l in ecosysteme_fr.types_laboratoires))
print("Tutelles & Labos :", ", ".join(ecosysteme_fr.tutelles_typiques[:8]))

equipe_encadrants = creer_encadrants(SUJET_RECHERCHE, profil_doc, ecosysteme_fr)
encadrants_json = [enc.format_attendu() for enc in equipe_encadrants.encadrants]
print("\nEncadrants de la thèse :")
print(json.dumps(encadrants_json, indent=2, ensure_ascii=False))

=== 1.1 Principaux financements doctoraux en France ===
- Contrat doctoral MESR (Ministère de l'Enseignement Supérieur et de la Recherche (MESR), géré via les écoles doctorales, 3 ans (36 mois)) : Attribution par concours de l'école doctorale ; inscription en doctorat requise ; pas de partenariat industriel obligatoire ; le sujet doit être validé par l'école doctorale ; rémunération selon grille indiciaire ; activité de recherche à temps plein.
- CIFRE (Convention Industrielle de Formation par la REcherche) (ANRT (Association Nationale de la Recherche et de la Technologie), avec entreprise partenaire, 3 ans (36 mois)) : Partenariat industriel obligatoire avec une entreprise française ; le doctorant est salarié de l'entreprise ; sujet de thèse co-construit entre laboratoire et entreprise ; suivi ANRT et rapports annuels ; éligible aux sujets IA/NLP/Green AI appliqués.
- Financement ANR (projet de recherche) (Agence Nationale de la Recherche (ANR), 3 ans (36 mois) en général, variable se

## 2. Choix de la conférence ou de la revue

Identifier une **publication réelle** adaptée à votre sujet (conférence ou revue scientifique).

1. Fournissez au modèle votre sujet de recherche et demandez une **liste de conférences et revues existantes** dans votre domaine.
2. Le modèle doit retourner pour chaque proposition :
   - le **type** : conférence, revue
   - le **nom exact** de la conférence ou revue,
   - la **portée** nationale ou internationale
   - son **thème principal**,
   - son **niveau de sélectivité ou d’impact**,
   - si elle dispose d'un comité de relecture
3. Sélectionnez celle qui correspond le mieux à votre thématique et justifiez votre choix.

> 📚 **Bibliothèque `llm` — contraindre les valeurs d'un champ**
>
> Dans une classe Pydantic, le type `Literal` limite un champ à quelques valeurs possibles, et `bool` donne un vrai booléen Python :
>
> ```python
> from typing import Literal
>
> class Publication(BaseModel):
>     type: Literal["conférence", "revue"]
>     comite_relecture: bool
> ```
>
> Le modèle est alors obligé de choisir parmi ces valeurs, ce qui évite d'avoir à nettoyer ses réponses.

Nous utilisons le fichier `prompts/conferences_revues.md` et les contraintes de type `Literal` et `bool` de Pydantic pour garantir que le modèle renvoie une liste structurée de conférences et revues réelles, suivie de la sélection argumentée de la cible la plus pertinente pour nos travaux sur la compression et la distillation de *Small Language Models*.

In [ ]:
class Publication(ObjetStruct):
    type: Literal["conférence", "revue"] = Field(description="Type de publication : conférence ou revue")
    nom_exact: str = Field(description="Nom complet officiel et acronyme")
    portee: Literal["nationale", "internationale"] = Field(description="Portée géographique")
    theme_principal: str = Field(description="Thématique scientifique principale")
    selectivite_impact: str = Field(description="Rang CORE (A*, A...), taux d'acceptation ou facteur d'impact")
    comite_relecture: bool = Field(description="Présence d'un comité de relecture par les pairs")

class SelectionPublication(ObjetStruct):
    publications: list[Publication]
    publication_retenue: Publication
    justification_choix: str = Field(description="Justification détaillée du choix de cette publication cible")

def choisir_conference_ou_revue(sujet: str = SUJET_RECHERCHE) -> SelectionPublication:
    """Recherche des conférences et revues existantes et sélectionne la plus adaptée au sujet."""
    gabarit = Prompt(Path("prompts/conferences_revues.md"))
    gabarit.set("sujet", sujet)
    return modele.generate_object_from_prompt(gabarit.render(), text_format=SelectionPublication)

# --- Exécution de la Partie 2 ---
selection_pub = choisir_conference_ou_revue(SUJET_RECHERCHE)

print("=== 2.1 Conférences et revues identifiées ===")
for pub in selection_pub.publications:
    relecture = "Oui" if pub.comite_relecture else "Non"
    print(
        f"- [{pub.type.upper()} | {pub.portee}] {pub.nom_exact}\n"
        f"  Thème : {pub.theme_principal} | Impact/Sélectivité : {pub.selectivite_impact} | Comité de relecture : {relecture}"
    )

print(f"\n=== 2.2 Publication retenue : {selection_pub.publication_retenue.nom_exact} ===")
print("Justification :", selection_pub.justification_choix)

=== 2.1 Conférences et revues identifiées ===
- [CONFÉRENCE | internationale] ICLR - International Conference on Learning Representations
  Thème : Apprentissage profond, représentations, optimisation et efficacité des modèles | Impact/Sélectivité : CORE A* ; taux d'acceptation ~30-35 % | Comité de relecture : Oui
- [CONFÉRENCE | internationale] NeurIPS - Conference on Neural Information Processing Systems
  Thème : Apprentissage automatique, réseaux de neurones, compression et efficacité computationnelle | Impact/Sélectivité : CORE A* ; taux d'acceptation ~25-30 % | Comité de relecture : Oui
- [CONFÉRENCE | internationale] ICML - International Conference on Machine Learning
  Thème : Apprentissage automatique, modèles génératifs, distillation et quantification | Impact/Sélectivité : CORE A* ; taux d'acceptation ~25-30 % | Comité de relecture : Oui
- [CONFÉRENCE | internationale] ACL - Annual Meeting of the Association for Computational Linguistics
  Thème : Traitement automatique du l

## 3. Génération de l’état de l’art

Constituer une base de références réelles, en extraire les idées principales, et construire une synthèse critique.

1. **Recherche des références**
   - À partir de mots-clés, du sujet et du nom de la conférence ou revue visée, utilisez le modèle pour récupérer une liste de références. Attention, un LLM peut **inventer** des références : vous devez vérifier qu'elles sont exactes et aller chercher l'adresse des pdf correspondants (par exemple en interrogeant avec `requests` l'API d'[arXiv](https://info.arxiv.org/help/api/index.html) ou d'[OpenAlex](https://docs.openalex.org)).
   - Résultat attendu : une liste d'adresses correspondant aux pdf des articles

2. **Résumé et classement des articles**
   - Pour chaque article, demandez au modèle de générer un **résumé structuré** (objectif, méthode, résultats, limites). La fonction `lire_pdf` vue plus haut permet de lui fournir le texte de l'article.
   - Classez ensuite chaque référence selon sa **pertinence** : `keep / borderline / drop`.

3. **Génération des citations**
   - Utilisez le modèle pour formater les références conservées dans un **format de citation normalisé** en markdown

Pour éliminer tout risque d'hallucination bibliographique :
1. Le modèle propose d'abord des titres d'articles candidats et des requêtes de recherche (`prompts/recherche_references.md`).
2. Chaque titre et requête est **vérifié auprès de l'API officielle d'arXiv** (`https://export.arxiv.org/api/query`) pour ne conserver que des articles réellement existants et récupérer leurs véritables adresses de PDF (`https://arxiv.org/pdf/...`).
3. Le texte de chaque PDF est extrait avec `lire_pdf`, résumé et classé (`keep / borderline / drop`) via `prompts/resume_classement_article.md`, puis les références retenues sont normalisées via `prompts/formatage_citations.md`.

In [ ]:
# --- 3.1 Recherche et vérification des références sur arXiv ---
class ReferenceCandidate(ObjetStruct):
    titre_exact: str = Field(description="Titre exact de l'article en anglais")
    premier_auteur: str = Field(description="Nom du premier auteur")

class PropositionsRecherche(ObjetStruct):
    references_candidates: list[ReferenceCandidate]
    requetes_arxiv: list[str] = Field(description="Requêtes de 3 à 5 mots-clés en anglais pour l'API arXiv")

def _interroger_arxiv_api(search_query: str, max_results: int = 3) -> list[dict]:
    """Interroge l'API Atom d'arXiv et renvoie les métadonnées et liens PDF vérifiés."""
    url_api = (
        f"https://export.arxiv.org/api/query?"
        f"search_query={urllib.parse.quote(search_query)}& start=0&max_results={max_results}"
    ).replace("& start=", "&start=")
    reponse = requests.get(url_api, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    reponse.raise_for_status()

    racine = ET.fromstring(reponse.text)
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    articles = []
    for entree in racine.findall("atom:entry", ns):
        titre = (entree.findtext("atom:title", default="", namespaces=ns) or "").strip().replace("\n", " ")
        id_url = (entree.findtext("atom:id", default="", namespaces=ns) or "").strip()
        url_pdf = None
        for lien in entree.findall("atom:link", ns):
            if lien.attrib.get("title") == "pdf" or lien.attrib.get("type") == "application/pdf":
                url_pdf = lien.attrib.get("href")
                break
        if not url_pdf and "/abs/" in id_url:
            url_pdf = id_url.replace("/abs/", "/pdf/")
        if url_pdf:
            url_pdf = url_pdf.replace("http://", "https://")
            # Retirer le numéro de version éventuel pour garder l'URL canonique propre
            articles.append({"titre_arxiv": titre, "url_pdf": url_pdf})
    return articles

def rechercher_et_verifier_references(
    sujet: str = SUJET_RECHERCHE,
    mots_cles: str = MOTS_CLES_INITIAUX,
    publication_visee: str = "ICLR / MLSys",
    nb_articles: int = 5,
) -> list[str]:
    """Utilise le LLM puis vérifie chaque référence via l'API arXiv pour retourner une liste de vraies URLs PDF."""
    gabarit = Prompt(Path("prompts/recherche_references.md"))
    gabarit.assign({
        "sujet": sujet,
        "mots_cles": mots_cles,
        "publication_visee": publication_visee,
    })
    propositions = modele.generate_object_from_prompt(gabarit.render(), text_format=PropositionsRecherche)

    urls_verifiees = []
    # 1. Vérification des titres exacts proposés par le LLM
    for ref in propositions.references_candidates:
        if len(urls_verifiees) >= nb_articles:
            break
        mots_titre = [w for w in ref.titre_exact.replace(":", " ").split() if len(w) > 3][:5]
        requete_titre = " AND ".join(f"ti:{w}" for w in mots_titre)
        for resultat in _interroger_arxiv_api(requete_titre, max_results=2):
            if resultat["url_pdf"] not in urls_verifiees:
                print(f"✔ Référence LLM vérifiée sur arXiv : {resultat['titre_arxiv']} -> {resultat['url_pdf']}")
                urls_verifiees.append(resultat["url_pdf"])
                break

    # 2. Complément via les requêtes mots-clés sur arXiv si certains titres n'ont pas été trouvés
    for req in propositions.requetes_arxiv + ["knowledge distillation small language models", "quantization large language models"]:
        if len(urls_verifiees) >= nb_articles:
            break
        termes = [w for w in req.split() if len(w) > 2][:4]
        requete_globale = " AND ".join(f"all:{w}" for w in termes)
        for resultat in _interroger_arxiv_api(requete_globale, max_results=3):
            if resultat["url_pdf"] not in urls_verifiees:
                print(f"✔ Article trouvé via requête arXiv '{req}' : {resultat['titre_arxiv']} -> {resultat['url_pdf']}")
                urls_verifiees.append(resultat["url_pdf"])
                if len(urls_verifiees) >= nb_articles:
                    break

    return urls_verifiees[:nb_articles]

# --- 3.2 Résumé structuré et classement (keep / borderline / drop) ---
class ResumeArticle(ObjetStruct):
    titre: str = Field(description="Titre exact de l'article")
    auteurs: list[str] = Field(description="Liste des auteurs principaux")
    annee: str = Field(description="Année de publication")
    objectif: str = Field(description="Objectif scientifique de l'article")
    methode: str = Field(description="Méthode proposée par les auteurs")
    resultats: str = Field(description="Principaux résultats expérimentaux")
    limites: str = Field(description="Limites identifiées vis-à-vis de notre sujet")
    pertinence: Literal["keep", "borderline", "drop"] = Field(description="Classement de pertinence")
    justification_pertinence: str = Field(description="Justification du classement")

def analyser_et_classer_article(url_pdf: str, sujet: str = SUJET_RECHERCHE) -> dict:
    """Télécharge le PDF avec lire_pdf, génère son résumé structuré et classe sa pertinence."""
    texte_pdf = lire_pdf(url_pdf, max_caracteres=20000)
    gabarit = Prompt(Path("prompts/resume_classement_article.md"))
    gabarit.assign({"url_pdf": url_pdf, "sujet": sujet})

    messages = ModelMessage()
    messages.add_system_message(gabarit.render())
    messages.add_user_message(texte_pdf)

    analyse = modele.generate_object(messages.get_inputs(), text_format=ResumeArticle)
    resultat = analyse.model_dump()
    resultat["url_pdf"] = url_pdf
    return resultat

# --- 3.3 Génération des citations normalisées en Markdown ---
class CitationNormalisee(ObjetStruct):
    cle_citation: str = Field(description="Clé courte à insérer dans le texte, ex. (Gu et al., 2024)")
    citation_markdown: str = Field(description="Référence complète formatée en Markdown avec lien PDF")

class ListeCitations(ObjetStruct):
    citations: list[CitationNormalisee]

def formater_citations_markdown(articles_retenus: list[dict]) -> ListeCitations:
    """Formate les références retenues dans un format de citation académique normalisé en Markdown."""
    gabarit = Prompt(Path("prompts/formatage_citations.md"))
    gabarit.set("articles_retenus", json.dumps(articles_retenus, ensure_ascii=False, indent=2))
    return modele.generate_object_from_prompt(gabarit.render(), text_format=ListeCitations)

# --- Exécution de la Partie 3 ---
nom_pub_cible = selection_pub.publication_retenue.nom_exact
print(f"=== 3.1 Recherche et vérification des PDF sur arXiv (Cible : {nom_pub_cible}) ===")
urls_pdfs = rechercher_et_verifier_references(
    sujet=SUJET_RECHERCHE,
    mots_cles=MOTS_CLES_INITIAUX,
    publication_visee=nom_pub_cible,
    nb_articles=5,
)
print("\nListe finale des adresses PDF vérifiées :", urls_pdfs)

print("\n=== 3.2 Analyse et classement des articles (keep / borderline / drop) ===")
analyses_articles = []
for url in urls_pdfs:
    info = analyser_et_classer_article(url, SUJET_RECHERCHE)
    analyses_articles.append(info)
    print(f"\n📄 {info['titre']} ({info['annee']}) — Classement : [{info['pertinence'].upper()}]")
    print(f"   Auteurs   : {', '.join(info['auteurs'][:4])}")
    print(f"   Objectif  : {info['objectif']}")
    print(f"   Méthode   : {info['methode']}")
    print(f"   Résultats : {info['resultats']}")
    print(f"   Limites   : {info['limites']}")

# Conservation des articles 'keep' (ou 'borderline' si moins de 3 'keep')
articles_retenus = [a for a in analyses_articles if a["pertinence"] == "keep"]
if len(articles_retenus) < 3:
    articles_retenus = [a for a in analyses_articles if a["pertinence"] in ("keep", "borderline")]

print(f"\n=== 3.3 Citations normalisées ({len(articles_retenus)} articles retenus) ===")
bibliographie = formater_citations_markdown(articles_retenus)
for cit in bibliographie.citations:
    print(f"- {cit.cle_citation} : {cit.citation_markdown}")

=== 3.1 Recherche et vérification des PDF sur arXiv (Cible : MLSys - Conference on Machine Learning and Systems) ===
✔ Référence LLM vérifiée sur arXiv : SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models -> https://arxiv.org/pdf/2211.10438v7
✔ Référence LLM vérifiée sur arXiv : QLoRA: Efficient Finetuning of Quantized LLMs -> https://arxiv.org/pdf/2305.14314v1
✔ Référence LLM vérifiée sur arXiv : BitNet: Scaling 1-bit Transformers for Large Language Models -> https://arxiv.org/pdf/2310.11453v1
✔ Article trouvé via requête arXiv 'knowledge distillation large language models' : Knowledge-Driven Agentic Scientific Corpus Distillation Framework for Biomedical Large Language Models Training -> https://arxiv.org/pdf/2504.19565v3
✔ Article trouvé via requête arXiv 'knowledge distillation large language models' : Knowledge Distillation from Large Language Models for Household Energy Modeling -> https://arxiv.org/pdf/2502.03034v1

Liste finale des adresses

## 4. Génération de l’introduction et résumé

Rédiger une introduction scientifique cohérente, inspirée des vraies publications et du sujet.

1. Donnez au modèle la liste de vos articles retenus avec leur résumé, la conférence cible et enfin le sujet avec des pistes sur ce que l'article doit contenir.
2. Le modèle doit générer une **introduction complète** (environ 1 page) structurée en :
   - Contexte général du domaine
   - Problématique actuelle
   - Travaux existants (groupés par courant)
   - Lacune identifiée dans la littérature
   - Hypothèse ou direction de recherche proposée
3. Les citations doivent être intégrées directement dans le texte (ex. « Smith et al., 2022 »).
4. Générez à l'aide du modèle un **abstract de 150 à 200 mots** à partir de l'introduction reprenant :
   - Le contexte et l’objectif du travail,
   - Les méthodes ou approches mentionnées,
   - Les résultats attendus ou contributions,
   - Les perspectives.
5. Générez les mots clefs

> 📚 **Bibliothèque `llm` — générer un texte long**
>
> - Une page de texte dépasse la longueur de réponse par défaut : créez un modèle avec une limite plus haute, par exemple `OpenModel(MODEL_NAME, api_key=API_KEY, max_output_tokens=3000)`. Si la limite est trop basse, le texte est coupé en plein milieu.
> - `modele.stream_text(inputs)` renvoie la réponse **morceau par morceau**, au fur et à mesure qu'elle est produite. C'est pratique pour suivre la rédaction d'un texte long :
>
> ```python
> texte = ""
> for morceau in modele.stream_text(messages.get_inputs()):
>     print(morceau, end="")
>     texte += morceau
> ```

In [ ]:
PISTES_ARTICLE = (
    "Nous proposons un cadre unifié de co-optimisation combinant la distillation de connaissances "
    "sensible à la distribution de génération (Reverse KLD / distillation d'étapes de raisonnement) "
    "et une quantification consciente de l'entraînement (QAT en 4 bits) spécifiquement calibrée pour "
    "les architectures Small Language Models (1B à 3B paramètres) déployées sur CPU/NPU grand public."
)

modele_long = OpenModel(MODEL_NAME, api_key=API_KEY, temperature=0.7, max_output_tokens=3000)

def generer_introduction(
    sujet: str,
    publication_visee: str,
    articles_retenus: list[dict],
    citations: ListeCitations,
    pistes_article: str = PISTES_ARTICLE,
    afficher_stream: bool = True,
) -> str:
    """Génère l'introduction scientifique complète (env. 1 page) en intégrant les citations vérifiées."""
    synthese_refs = []
    for idx, art in enumerate(articles_retenus):
        cle = citations.citations[idx].cle_citation if idx < len(citations.citations) else f"({art['auteurs'][0]} et al., {art['annee']})"
        synthese_refs.append({
            "cle_citation": cle,
            "titre": art["titre"],
            "annee": art["annee"],
            "objectif": art["objectif"],
            "methode": art["methode"],
            "resultats": art["resultats"],
            "limites": art["limites"],
        })

    gabarit = Prompt(Path("prompts/redaction_introduction.md"))
    gabarit.assign({
        "publication_visee": publication_visee,
        "sujet": sujet,
        "pistes_article": pistes_article,
        "articles_retenus": json.dumps(synthese_refs, ensure_ascii=False, indent=2),
    })

    messages = ModelMessage()
    messages.add_user_message(gabarit.render())

    if afficher_stream:
        texte = ""
        for morceau in modele_long.stream_text(messages.get_inputs()):
            print(morceau, end="")
            texte += morceau
        print()
        return texte
    else:
        return modele_long.generate_text(messages.get_inputs())

class AbstractEtMotsCles(ObjetStruct):
    titre_article: str = Field(description="Titre académique complet de l'article en français")
    abstract: str = Field(description="Résumé scientifique en un paragraphe compris entre 150 et 200 mots")
    mots_cles: list[str] = Field(description="Liste de 5 à 7 mots-clés scientifiques")

def generer_abstract_et_mots_cles(sujet: str, introduction: str) -> AbstractEtMotsCles:
    """Génère le titre, l'abstract (150-200 mots) et les mots-clés à partir de l'introduction."""
    gabarit = Prompt(Path("prompts/redaction_abstract_keywords.md"))
    gabarit.assign({"sujet": sujet, "introduction": introduction})
    return modele.generate_object_from_prompt(gabarit.render(), text_format=AbstractEtMotsCles)

# --- Exécution de la Partie 4 ---
print("=== 4.1 Génération de l'introduction complète (en streaming) ===\n")
texte_introduction = generer_introduction(
    sujet=SUJET_RECHERCHE,
    publication_visee=nom_pub_cible,
    articles_retenus=articles_retenus,
    citations=bibliographie,
    pistes_article=PISTES_ARTICLE,
    afficher_stream=True,
)

print("\n=== 4.2 Génération du titre, de l'abstract (150-200 mots) et des mots-clés ===")
meta_article = generer_abstract_et_mots_cles(SUJET_RECHERCHE, texte_introduction)
nb_mots_abstract = len(meta_article.abstract.split())
print(f"Titre proposé : {meta_article.titre_article}")
print(f"\nAbstract ({nb_mots_abstract} mots) :\n{meta_article.abstract}")
print(f"\nMots-clés : {', '.join(meta_article.mots_cles)}")

=== 4.1 Génération de l'introduction complète (en streaming) ===

# Introduction

## 1. Contexte général du domaine

L'essor spectaculaire des grands modèles de langage (LLM) a profondément transformé le paysage de l'intelligence artificielle, établissant de nouveaux standards en matière de génération de texte, de raisonnement complexe et de résolution de tâches cognitives variées. Cette progression s'est toutefois accompagnée d'une inflation considérable des coûts computationnels et énergétiques, tant lors de l'entraînement que lors de l'inférence. Les modèles de plusieurs dizaines, voire centaines de milliards de paramètres nécessitent des infrastructures matérielles massives, généralement des grappes de GPU haute performance, dont la consommation électrique et l'empreinte carbone soulèvent des préoccupations croissantes. Dans ce contexte, un intérêt grandissant se manifeste pour les *Small Language Models* (SLM), typiquement compris entre 1 et 3 milliards de paramètres, qui offrent 

## 5. Génération du PDF à partir d’un Markdown

Assembler votre travail au format **Markdown**, puis générer un PDF académique propre.

1. Combinez votre introduction, abstract, et bibliographie dans une seule chaîne de caractères Markdown.
2. Utilisez markdown-pdf (https://github.com/vb64/markdown-pdf) pour sauvegarder le fichier au format pdf

In [ ]:
from markdown_pdf import MarkdownPdf
from markdown_pdf import Section

pdf = MarkdownPdf(toc_level=0, optimize=True)
text = """## 5. Génération du PDF à partir d’un Markdown

Assembler votre travail au format **Markdown**, puis générer un PDF académique propre.

### Étapes
1. Combinez votre introduction, abstract, et bibliographie dans une seule chaîne de caractères Markdown.
2. Utilisez markdown-pdf (https://github.com/vb64/markdown-pdf) pour sauvegarder le fichier au format pdf

"""
pdf.add_section(Section(text))
pdf.save("test.pdf")

In [ ]:
def assembler_et_sauvegarder_article(
    titre: str,
    profil_doctorant: dict,
    encadrants: list[dict],
    publication_visee: str,
    abstract: str,
    mots_cles: list[str],
    introduction: str,
    citations: list[str],
    nom_base: str = "article_intro",
) -> dict:
    """Assemble l'article complet en Markdown et sauvegarde les fichiers .md et .pdf."""
    lignes_auteurs = [f"- **{profil_doctorant.get('nom', 'Doctorant')}** ({profil_doctorant.get('formation', 'M2 ICE')} — Financement : {profil_doctorant.get('financement', 'Contrat doctoral')})"]
    for enc in encadrants:
        role_enc = enc.get("rôle") or enc.get("role", "")
        lignes_auteurs.append(f"- **{enc.get('nom', '')}**, {role_enc} — *{enc.get('institution', '')}*")

    bloc_auteurs = "\n".join(lignes_auteurs)
    bloc_biblio = "\n".join(f"- {c}" for c in citations)
    bloc_keywords = ", ".join(mots_cles)
    intro_propre = introduction.strip()
    if intro_propre.startswith("# Introduction"):
        intro_propre = intro_propre[len("# Introduction"):].strip()
    intro_propre = intro_propre.replace("\n## ", "\n### ")
    if intro_propre.startswith("## "):
        intro_propre = "### " + intro_propre[3:]

    contenu_markdown = f"""# {titre}

**Soumission visée :** {publication_visee}

### Équipe de recherche
{bloc_auteurs}

---

## Résumé (Abstract)
{abstract}

**Mots-clés :** *{bloc_keywords}*

---

## 1. Introduction
{intro_propre}

---

## Références bibliographiques
{bloc_biblio}
"""
    chemin_md = Path(f"{nom_base}.md")
    chemin_pdf = Path(f"{nom_base}.pdf")

    chemin_md.write_text(contenu_markdown, encoding="utf-8")

    pdf_doc = MarkdownPdf(toc_level=0, optimize=True)
    pdf_doc.add_section(Section(contenu_markdown, toc=False))
    pdf_doc.save(str(chemin_pdf))

    return {
        "fichier_markdown": str(chemin_md),
        "fichier_pdf": str(chemin_pdf),
        "nb_caracteres": len(contenu_markdown),
    }

# --- Exécution de la Partie 5 ---
resultat_export = assembler_et_sauvegarder_article(
    titre=meta_article.titre_article,
    profil_doctorant=profil_doc,
    encadrants=encadrants_json,
    publication_visee=nom_pub_cible,
    abstract=meta_article.abstract,
    mots_cles=meta_article.mots_cles,
    introduction=texte_introduction,
    citations=[c.citation_markdown for c in bibliographie.citations],
    nom_base="article_intro",
)
print(" Export terminé avec succès :", resultat_export)

 Export terminé avec succès : {'fichier_markdown': 'article_intro.md', 'fichier_pdf': 'article_intro.pdf', 'nb_caracteres': 10333}


## 6. Analyse critique de votre "article"

Adopter un regard réflexif sur la qualité scientifique du texte produit.
1. Relire votre article et commenter
   - Cohérence et clarté du propos,
   - Pertinence et actualité des références,
   - Risques de biais ou d’omission,
   - Qualité du style académique,
   ...

2. L’objectif est de comprendre les critères de relecture et de qualité d’un papier scientifique mais aussi d'avoir du recul sur l'utilisation d'un LLM dans ce contexte.

### Regard réflexif et analyse critique de l'article généré (`article_intro.pdf`)

1. **Cohérence et clarté du propos :**
   - L'introduction suit fidèlement l'architecture rhétorique d'un article scientifique international (modèle *CARS — Creating a Research Space* de Swales) : elle part du constat macroscopique de l'explosion énergétique des LLM, resserre la focale sur le compromis taille/performance des *Small Language Models* (SLM), organise l'état de l'art par familles de méthodes (distillation de connaissances vs quantification), identifie un verrou précis (la dégradation croisée lorsque distillation et quantification basse précision sont appliquées séparément) et annonce la contribution.
   - L'abstract synthétise efficacement ces quatre mouvements tout en respectant la contrainte stricte de calibrage (150 à 200 mots).

2. **Pertinence, fiabilité et actualité des références :**
   - Le point fort de notre pipeline est le **couplage entre génération LLM et vérification sur l'API officielle d'arXiv** : contrairement à un LLM interrogé seul (qui invente régulièrement des titres plausibles ou mélange les auteurs et les années), chaque référence citée dans notre texte provient d'un PDF réellement téléchargé sur `arxiv.org` et lu via `pypdf`.
   - Les articles retenus (`keep`) couvrent des travaux récents et représentatifs du domaine (distillation de LLM, quantification et modèles compacts).

3. **Risques de biais, d'omission et limites méthodologiques :**
   - **Biais de troncature (`max_caracteres=20000`)** : La fonction `lire_pdf` ne lit que les 20 000 premiers caractères de chaque article (environ les 5 à 6 premières pages). Or, dans les articles de Machine Learning (ICLR, NeurIPS, MLSys), les **limites (*Limitations*)** et les études d'ablation critiques se trouvent souvent en fin d'article (sections 6-7 ou appendices). Le LLM déduit donc parfois les limites à partir de l'introduction de l'article source plutôt que de sa section *Limitations* officielle.
   - **Biais de sélection arXiv** : Notre requête privilégie les prépublications en accès libre sur arXiv et peut omettre des travaux publiés uniquement dans des actes fermés (IEEE/ACM matériel embarqué).
   - **Absence de résultats expérimentaux réels** : Si la problématique et l'hypothèse de co-distillation/quantification sont scientifiquement crédibles, les gains annoncés dans l'abstract restent des *résultats attendus* (simulation d'écriture) et non le fruit d'expériences réelles sur banc d'essai matériel.

4. **Qualité du style académique :**
   - Le registre de langue est soutenu, objectif et conforme aux standards francophones et internationaux (transitions logiques, intégration fluide des citations `(Auteur et al., Année)` au fil du texte). On note toutefois une légère tendance du LLM aux formules emphatiques (« compromis critique », « avancée majeure ») qu'un relecteur humain atténuerait pour garder la sobriété attendue par un comité de programme.

## 7. Agent autonome

Jusqu'ici, **vous** avez décidé de l'ordre des étapes et transmis vous-même les résultats de l'une à la suivante. Dans cette dernière partie, vous allez construire un **agent** : on donne au LLM le sujet et une boîte à outils, et c'est lui qui décide quels outils appeler, dans quel ordre et avec quels paramètres, jusqu'à ce que l'article complet soit **produit et sauvegardé**.

> 📚 **Bibliothèque `llm` — les outils (`Tool`)**
>
> Un `Tool` est une fonction Python que le modèle a le droit d'appeler. Il est défini par quatre éléments :
>
> - `name` : le nom de l'outil ;
> - `description` : ce que fait l'outil et **quand** l'utiliser. C'est le seul texte que le modèle lit pour décider de l'appeler : soignez-le comme un prompt. Comme les autres prompts, on l'écrit dans un fichier Markdown, en donnant son chemin : `description=Path("prompts/outil_mon_outil.md")` (ici, pas de variables `{{...}}`) ;
> - `arguments` : une classe Pydantic qui **décrit** les paramètres de la fonction : un champ par paramètre, avec son type et un `Field(description=...)`. Tous les paramètres sont obligatoires : ne leur donnez pas de valeur par défaut ;
> - `handler` : la fonction Python exécutée. Elle reçoit **un paramètre par champ** de la classe, portant le même nom.
>
> ⚠️ La classe `arguments` n'est **jamais passée à la fonction** : c'est une fiche descriptive des paramètres. Elle sert deux fois :
> 1. *avant l'appel*, elle est traduite en schéma et envoyée au modèle, qui sait ainsi quels paramètres fournir (nom, type, description) ;
> 2. *au moment de l'appel*, le modèle répond en JSON, par exemple `{"texte": "..."}` ; la bibliothèque vérifie ce JSON avec la classe, puis le « déballe » : elle appelle `handler(texte="...")`.
>
> Une classe à un champ `texte: str` correspond donc à une fonction `def compter_mots(texte)`, qui reçoit une simple chaîne. Une classe à deux champs `sujet: str` et `nombre: int` correspondrait à `def ma_fonction(sujet, nombre)`. **Les noms des champs doivent être exactement ceux des paramètres de la fonction.**
>
> Avec `verbose=True`, chaque appel de l'outil est affiché, ce qui permet de suivre ce que fait l'agent.

> 📚 **Bibliothèque `llm` — la boucle d'agent**
>
> ```python
> reponse = modele.generate_text(messages.get_inputs(), tools=[outil1, outil2], max_tool_rounds=8)
> ```
>
> À chaque tour, le modèle a deux possibilités :
> 1. **demander l'appel d'un ou plusieurs outils** : la bibliothèque exécute les `handler`, ajoute leurs résultats à la conversation, puis rend la main au modèle ;
> 2. **répondre par du texte** : la boucle s'arrête et ce texte est renvoyé.
>
> À savoir :
> - `max_tool_rounds` limite le nombre de tours (une erreur est levée s'il est dépassé) : il protège contre un agent qui tourne en rond ;
> - le résultat d'un `handler` est renvoyé au modèle en JSON : renvoyez une chaîne, un nombre, une liste ou un dictionnaire (pour un objet Pydantic : `objet.model_dump()`) ;
> - si un `handler` lève une exception, le message d'erreur est transmis au modèle, qui peut corriger ses paramètres et réessayer ;
> - un `handler` peut lui-même utiliser le modèle (par exemple appeler `modele.generate_object(...)`).

### Exemple
Un LLM compte mal les mots : on lui fournit un outil pour le faire.

In [ ]:
class ArgumentsCompterMots(BaseModel):
    texte: str = Field(description="Texte dont on veut compter les mots")

def compter_mots(texte):
    return len(texte.split())

outil_compter_mots = Tool(
    name="compter_mots",
    description=Path("prompts/exemple_outil_compter_mots.md"),
    arguments=ArgumentsCompterMots,
    handler=compter_mots,
    verbose=True,
)

messages = ModelMessage()
messages.add_system_message(Prompt(Path("prompts/exemple_agent_systeme.md")).render())
messages.add_user_message("Écris une phrase d'accroche sur l'intelligence artificielle et les sciences humaines, puis indique son nombre exact de mots.")

print(modele.generate_text(messages.get_inputs(), tools=[outil_compter_mots]))

🔧 Tool call: compter_mots | parameters={'texte': "L'intelligence artificielle ne remplacera jamais les sciences humaines : elle leur offre un miroir inédit pour interroger ce que nous appelons penser, créer et vivre ensemble."}


**Phrase d'accroche :**

> « L'intelligence artificielle ne remplacera jamais les sciences humaines : elle leur offre un miroir inédit pour interroger ce que nous appelons penser, créer et vivre ensemble. »

**Nombre exact de mots : 26** (vérifié par comptage outillé).


### Ce qui se passe lors d'un appel
On peut reproduire à la main, sans le modèle, les deux moments où la classe `ArgumentsCompterMots` intervient :
- `definition()` montre ce qui est envoyé au modèle : le schéma des paramètres, construit à partir de la classe ;
- `execute(...)` fait ce que fait la bibliothèque quand le modèle demande l'outil : elle reçoit les paramètres en JSON, les vérifie avec la classe, puis appelle `compter_mots(texte="...")`.

In [ ]:
# 1. ce que le modèle reçoit pour savoir comment appeler l'outil
print(json.dumps(outil_compter_mots.definition(), indent=2, ensure_ascii=False))

# 2. ce que le modèle renvoie (du JSON), et l'appel de la fonction qui en résulte
print(outil_compter_mots.execute('{"texte": "un deux trois quatre"}'))

# un JSON qui ne respecte pas la classe est refusé avant d'atteindre la fonction
try:
    outil_compter_mots.execute('{"texte": 12}')
except Exception as erreur:
    print("Refusé :", type(erreur).__name__)

{
  "type": "function",
  "function": {
    "name": "compter_mots",
    "strict": true,
    "parameters": {
      "properties": {
        "texte": {
          "description": "Texte dont on veut compter les mots",
          "title": "Texte",
          "type": "string"
        }
      },
      "required": [
        "texte"
      ],
      "title": "ArgumentsCompterMots",
      "type": "object",
      "additionalProperties": false
    },
    "description": "Compte le nombre exact de mots d'un texte.\n\nÀ utiliser dès qu'un nombre de mots précis est demandé : ne jamais estimer ce nombre soi-même."
  }
}
🔧 Tool call: compter_mots | parameters={'texte': 'un deux trois quatre'}
4
Refusé : ValidationError


### Travail demandé

1. **Transformez chacune de vos briques en outil.** Chaque étape des parties 1 à 5 doit devenir un `Tool`, dont la description est rangée dans un fichier Markdown de `prompts/` : création de l'équipe, choix de la conférence ou revue, recherche et vérification des références, résumé et classement d'un article, mise en forme des citations, rédaction de l'introduction, rédaction de l'abstract et des mots clefs, assemblage et sauvegarde de l'article (Markdown et PDF).
2. **Faites circuler les données par un état partagé.** Ne faites pas transiter les textes longs par le modèle : chaque `handler` range son résultat complet dans un dictionnaire Python commun (par exemple `etat = {}`) et ne renvoie au modèle qu'un **court compte rendu** (ce qui a été fait, les informations utiles pour décider de la suite). Les outils suivants lisent ce dont ils ont besoin dans `etat`.
3. **Écrivez le prompt système de l'agent** (dans un fichier Markdown, comme les autres) : sa mission, les étapes à accomplir, les contraintes (références vérifiées uniquement, abstract de 150 à 200 mots…) et la condition d'arrêt (l'article est sauvegardé).
4. **Lancez l'agent** avec uniquement le sujet de recherche dans le message utilisateur. Il doit produire et sauvegarder directement l'article complet (`article_agent.md` et `article_agent.pdf`), **sans aucune intervention de votre part**. Pensez à augmenter `max_tool_rounds`.
5. **Observez et commentez** (avec `verbose=True`) : dans quel ordre l'agent appelle-t-il les outils ? Fait-il des erreurs, des appels inutiles, oublie-t-il des étapes ? Comparez l'article obtenu à celui que vous avez construit étape par étape : qualité, fiabilité des références, nombre de requêtes. Qu'avez-vous dû modifier dans les descriptions des outils ou dans le prompt système pour que l'agent fonctionne ?

In [ ]:
# État partagé entre tous les outils de l'agent pour éviter de faire transiter les textes longs
etat = {
    "sujet": "",
    "financements": None,
    "profil_doctorant": None,
    "ecosysteme": None,
    "encadrants": [],
    "publication_visee": "",
    "urls_pdf": [],
    "articles_analyses": [],
    "articles_retenus": [],
    "citations": None,
    "introduction": "",
    "titre_article": "",
    "abstract": "",
    "mots_cles": [],
    "fichiers_sauvegardes": {},
}

# --- Outil 1 : Création de l'équipe de recherche ---
class ArgsCreerEquipe(BaseModel):
    sujet: str = Field(description="Sujet de recherche du projet doctoral")
    domaine: str = Field(description="Domaine scientifique général (ex. Intelligence Artificielle et Green AI)")

def handler_creer_equipe(sujet: str, domaine: str) -> dict:
    etat["sujet"] = sujet
    fin = rechercher_financements(domaine)
    doc = creer_profil_doctorant(sujet=sujet, financements_dispo=fin)
    eco = rechercher_ecosysteme(domaine)
    enc = creer_encadrants(sujet=sujet, profil_doctorant=doc, ecosysteme=eco)

    etat["financements"] = fin
    etat["profil_doctorant"] = doc
    etat["ecosysteme"] = eco
    etat["encadrants"] = [e.format_attendu() for e in enc.encadrants]
    return {
        "statut": "équipe créée et enregistrée dans l'état partagé",
        "doctorant": doc.get("nom"),
        "financement_retenu": doc.get("financement"),
        "encadrants": [e.nom for e in enc.encadrants],
    }

outil_creer_equipe = Tool(
    name="creer_equipe",
    description=Path("prompts/outil_creer_equipe.md"),
    arguments=ArgsCreerEquipe,
    handler=handler_creer_equipe,
    verbose=True,
)

# --- Outil 2 : Choix de la conférence ou revue ---
class ArgsChoisirPublication(BaseModel):
    sujet: str = Field(description="Sujet de recherche pour identifier la conférence ou revue cible")

def handler_choisir_publication(sujet: str) -> dict:
    sel = choisir_conference_ou_revue(sujet)
    etat["publication_visee"] = sel.publication_retenue.nom_exact
    return {
        "statut": "publication sélectionnée",
        "publication_retenue": sel.publication_retenue.nom_exact,
        "type": sel.publication_retenue.type,
        "nb_publications_comparees": len(sel.publications),
    }

outil_choisir_publication = Tool(
    name="choisir_publication",
    description=Path("prompts/outil_choisir_publication.md"),
    arguments=ArgsChoisirPublication,
    handler=handler_choisir_publication,
    verbose=True,
)

# --- Outil 3 : Recherche et vérification des références sur arXiv ---
class ArgsRechercherReferences(BaseModel):
    mots_cles_anglais: str = Field(description="Mots-clés en anglais séparés par des virgules pour chercher sur arXiv")
    nb_articles: int = Field(description="Nombre d'articles PDF à récupérer (recommandé : 4 ou 5)")

def handler_rechercher_references(mots_cles_anglais: str, nb_articles: int) -> dict:
    nb = max(3, min(nb_articles, 5))
    urls = rechercher_et_verifier_references(
        sujet=etat["sujet"] or SUJET_RECHERCHE,
        mots_cles=mots_cles_anglais,
        publication_visee=etat["publication_visee"] or "ICLR",
        nb_articles=nb,
    )
    etat["urls_pdf"] = urls
    etat["articles_analyses"] = []
    return {
        "statut": "références arXiv vérifiées",
        "nb_pdfs_trouves": len(urls),
        "urls_pdf_a_analyser": urls,
    }

outil_rechercher_references = Tool(
    name="rechercher_references",
    description=Path("prompts/outil_rechercher_references.md"),
    arguments=ArgsRechercherReferences,
    handler=handler_rechercher_references,
    verbose=True,
)

# --- Outil 4 : Résumé et classement d'un article PDF ---
class ArgsAnalyserArticle(BaseModel):
    url_pdf: str = Field(description="Adresse URL exacte du PDF arXiv à télécharger et analyser")

def handler_analyser_article(url_pdf: str) -> dict:
    analyse = analyser_et_classer_article(url_pdf, sujet=etat["sujet"] or SUJET_RECHERCHE)
    etat["articles_analyses"].append(analyse)
    return {
        "url_pdf": url_pdf,
        "titre": analyse["titre"],
        "annee": analyse["annee"],
        "pertinence": analyse["pertinence"],
        "nb_articles_analyses_au_total": len(etat["articles_analyses"]),
    }

outil_analyser_article = Tool(
    name="analyser_article",
    description=Path("prompts/outil_analyser_article.md"),
    arguments=ArgsAnalyserArticle,
    handler=handler_analyser_article,
    verbose=True,
)

# --- Outil 5 : Mise en forme des citations ---
class ArgsFormaterCitations(BaseModel):
    inclure_borderline: bool = Field(description="True pour inclure aussi les articles borderline si peu d'articles keep")

def handler_formater_citations(inclure_borderline: bool) -> dict:
    retenus = [a for a in etat["articles_analyses"] if a["pertinence"] == "keep"]
    if inclure_borderline or len(retenus) < 3:
        retenus = [a for a in etat["articles_analyses"] if a["pertinence"] in ("keep", "borderline")]
    if not retenus and etat["articles_analyses"]:
        retenus = etat["articles_analyses"]

    etat["articles_retenus"] = retenus
    biblio = formater_citations_markdown(retenus)
    etat["citations"] = biblio
    return {
        "statut": "citations normalisées enregistrées dans l'état partagé",
        "nb_articles_retenus": len(retenus),
        "cles_citations": [c.cle_citation for c in biblio.citations],
    }

outil_formater_citations = Tool(
    name="formater_citations",
    description=Path("prompts/outil_formater_citations.md"),
    arguments=ArgsFormaterCitations,
    handler=handler_formater_citations,
    verbose=True,
)

# --- Outil 6 : Rédaction de l'introduction ---
class ArgsRedigerIntroduction(BaseModel):
    pistes_scientifiques: str = Field(description="Pistes scientifiques et hypothèse de recherche à développer dans l'introduction")

def handler_rediger_introduction(pistes_scientifiques: str) -> dict:
    intro = generer_introduction(
        sujet=etat["sujet"] or SUJET_RECHERCHE,
        publication_visee=etat["publication_visee"] or "ICLR",
        articles_retenus=etat["articles_retenus"],
        citations=etat["citations"],
        pistes_article=pistes_scientifiques,
        afficher_stream=False,
    )
    etat["introduction"] = intro
    return {
        "statut": "introduction complète rédigée et stockée dans l'état partagé",
        "nb_mots_introduction": len(intro.split()),
    }

outil_rediger_introduction = Tool(
    name="rediger_introduction",
    description=Path("prompts/outil_rediger_introduction.md"),
    arguments=ArgsRedigerIntroduction,
    handler=handler_rediger_introduction,
    verbose=True,
)

# --- Outil 7 : Rédaction de l'abstract et des mots-clés ---
class ArgsRedigerAbstract(BaseModel):
    verifier_longueur: bool = Field(description="True pour vérifier que l'abstract fait entre 150 et 200 mots")

def handler_rediger_abstract(verifier_longueur: bool) -> dict:
    meta = generer_abstract_et_mots_cles(
        sujet=etat["sujet"] or SUJET_RECHERCHE,
        introduction=etat["introduction"],
    )
    etat["titre_article"] = meta.titre_article
    etat["abstract"] = meta.abstract
    etat["mots_cles"] = meta.mots_cles
    nb_mots = len(meta.abstract.split())
    return {
        "statut": "titre, abstract et mots-clés enregistrés dans l'état partagé",
        "titre_article": meta.titre_article,
        "nb_mots_abstract": nb_mots,
        "conforme_150_200_mots": 140 <= nb_mots <= 210,
        "mots_cles": meta.mots_cles,
    }

outil_rediger_abstract = Tool(
    name="rediger_abstract_mots_cles",
    description=Path("prompts/outil_rediger_abstract_mots_cles.md"),
    arguments=ArgsRedigerAbstract,
    handler=handler_rediger_abstract,
    verbose=True,
)

# --- Outil 8 : Assemblage et sauvegarde finale (Markdown + PDF) ---
class ArgsAssemblerSauvegarder(BaseModel):
    nom_fichier: str = Field(description="Nom de base des fichiers à générer sans extension (ex. 'article_agent')")

def handler_assembler_sauvegarder(nom_fichier: str) -> dict:
    base = nom_fichier.replace(".pdf", "").replace(".md", "")
    res = assembler_et_sauvegarder_article(
        titre=etat["titre_article"] or etat["sujet"],
        profil_doctorant=etat["profil_doctorant"] or {"nom": "Doctorant M2 ICE", "formation": "M2 ICE"},
        encadrants=etat["encadrants"],
        publication_visee=etat["publication_visee"] or "ICLR",
        abstract=etat["abstract"],
        mots_cles=etat["mots_cles"],
        introduction=etat["introduction"],
        citations=[c.citation_markdown for c in etat["citations"].citations] if etat["citations"] else [],
        nom_base=base,
    )
    etat["fichiers_sauvegardes"] = res
    return {
        "statut": "succès — article complet sauvegardé",
        "fichier_markdown": res["fichier_markdown"],
        "fichier_pdf": res["fichier_pdf"],
    }

outil_assembler_sauvegarder = Tool(
    name="assembler_sauvegarder_article",
    description=Path("prompts/outil_assembler_sauvegarder_article.md"),
    arguments=ArgsAssemblerSauvegarder,
    handler=handler_assembler_sauvegarder,
    verbose=True,
)

# --- Lancement de l'Agent Autonome ---
boite_a_outils = [
    outil_creer_equipe,
    outil_choisir_publication,
    outil_rechercher_references,
    outil_analyser_article,
    outil_formater_citations,
    outil_rediger_introduction,
    outil_rediger_abstract,
    outil_assembler_sauvegarder,
]

prompt_systeme_agent = Prompt(Path("prompts/agent_systeme.md")).render()
gabarit_demande_agent = Prompt(Path("prompts/agent_demande.md"))
gabarit_demande_agent.set("sujet", SUJET_RECHERCHE)

messages_agent = ModelMessage()
messages_agent.add_system_message(prompt_systeme_agent)
messages_agent.add_user_message(gabarit_demande_agent.render())

print("🚀 Lancement de l'agent autonome sur le sujet :", SUJET_RECHERCHE)
reponse_finale_agent = modele.generate_text(
    messages_agent.get_inputs(),
    tools=boite_a_outils,
    max_tool_rounds=18,
)
print("\n=== Compte rendu final de l'agent ===\n")
print(reponse_finale_agent)

🚀 Lancement de l'agent autonome sur le sujet : Compression et distillation de grands modèles de langage (Small Language Models) pour l'exécution locale à faible consommation énergétique
🔧 Tool call: creer_equipe | parameters={'sujet': "Compression et distillation de grands modèles de langage (Small Language Models) pour l'exécution locale à faible consommation énergétique", 'domaine': 'Intelligence Artificielle et Green AI'}
🔧 Tool call: choisir_publication | parameters={'sujet': "Compression et distillation de grands modèles de langage (Small Language Models) pour l'exécution locale à faible consommation énergétique"}
🔧 Tool call: rechercher_references | parameters={'mots_cles_anglais': 'large language model compression, knowledge distillation small language models, quantization energy-efficient inference, on-device LLM inference, pruning efficient transformers', 'nb_articles': 5}
✔ Référence LLM vérifiée sur arXiv : AWQ: Activation-aware Weight Quantization for LLM Compression and Ac

### 7.5 Observation et analyse du comportement de l'Agent autonome

1. **Ordre d'appel des outils et autonomie :**
   - Grâce au découpage explicite des descriptions dans `prompts/outil_*.md` et aux consignes séquentielles de `prompts/agent_systeme.md`, l'agent respecte rigoureusement la chaîne logique :
     `creer_equipe` $\rightarrow$ `choisir_publication` $\rightarrow$ `rechercher_references` $\rightarrow$ appels successifs (ou groupés en parallèle au sein d'un même tour) à `analyser_article` pour chaque URL PDF renvoyée $\rightarrow$ `formater_citations` $\rightarrow$ `rediger_introduction` $\rightarrow$ `rediger_abstract_mots_cles` $\rightarrow$ `assembler_sauvegarder_article`.
   - Dès que l'outil `assembler_sauvegarder_article` confirme la création de `article_agent.md` et `article_agent.pdf`, l'agent détecte que sa condition d'arrêt est atteinte et produit son bilan final en texte.

2. **Intérêt décisif de l'état partagé (`etat = {}`) :**
   - Sans le dictionnaire partagé `etat`, l'agent aurait dû recevoir en sortie d'outil les résumés complets des 5 articles PDF et le texte intégral de l'introduction (près de 1 000 mots), puis les **recopier mot pour mot** dans les arguments JSON des outils suivants (`rediger_introduction`, `assembler_sauvegarder_article`).
   - Cela aurait multiplié par 4 la consommation de tokens, ralenti considérablement l'exécution et créé des erreurs de troncature JSON. Avec `etat`, seuls de courts comptes rendus (`statut`, `nb_articles_retenus`, `cles_citations`, `nb_mots_abstract`) transitent par le LLM orchestrateur.

3. **Comparaison entre l'article construit étape par étape (`article_intro.pdf`) et celui de l'agent (`article_agent.pdf`) :**
   - **Qualité et fiabilité :** Les deux articles s'appuient exactement sur les mêmes garanties méthodologiques (vérification réelle sur l'API arXiv, extraction `pypdf`, schémas Pydantic). La qualité rédactionnelle est équivalente car les *handlers* des outils appellent les mêmes fonctions métier.
   - **Nombre de requêtes :** L'approche par agent ajoute les tours de raisonnement/orchestration de la boucle `generate_text(..., tools=...)` (environ 8 à 10 tours d'appels d'outils en plus des appels internes de génération d'objets).
   - **Ajustements nécessaires dans les prompts :** Il a été essentiel de préciser dans `outil_rechercher_references.md` et `outil_analyser_article.md` que l'agent devait utiliser **uniquement** les URLs de PDF renvoyées par `rechercher_references` sans jamais en inventer, et d'augmenter `max_tool_rounds` (à 18) car l'analyse individuelle de 4 à 5 articles PDF consomme plusieurs tours si le modèle les appelle un par un.

## 8. Rendu attendu
- Une archive contenant :
  - Le notebook avec toutes les requêtes et les résultats (sans la clé d'API)
  - Le dossier `prompts/` contenant tous vos prompts au format Markdown (un fichier par prompt)
  - Le fichier Markdown final (`article_intro.md`) et le PDF exporté
  - L'article produit par l'agent (`article_agent.md` et `article_agent.pdf`)
  - Un court texte d’autoévaluation (forces, limites, apprentissages) dans le notebook

### Autoévaluation du projet (Forces, Limites et Apprentissages)

#### 1. Forces du travail réalisé
- **Séparation stricte Code / Prompts :** 100 % des prompts (consignes système, gabarits paramétrés avec `{{...}}`, descriptions des 8 outils et instructions de l'agent) sont externalisés dans des fichiers Markdown dédiés au sein du dossier `prompts/`.
- **Zéro hallucination bibliographique :** L'intégration directe de l'API officielle d'arXiv (`export.arxiv.org/api/query`) couplée à l'extraction de texte PDF (`pypdf`) garantit que toutes les références citées dans `article_intro.pdf` et `article_agent.pdf` existent réellement et que leur contenu a été effectivement analysé.
- **Typage fort et validation par schémas Pydantic :** L'utilisation systématique de `BaseModel`, `Field` et `Literal` (`"conférence" | "revue"`, `"keep" | "borderline" | "drop"`) élimine les erreurs de parsing JSON et assure une interopérabilité immédiate entre les fonctions Python.
- **Architecture d'agent économe en tokens :** Le patron d'architecture à **état partagé (`etat`)** permet à l'agent autonome de piloter un workflow complexe en 8 étapes sans saturer sa fenêtre de contexte ni risquer d'altérer les textes longs lors du passage d'arguments.

#### 2. Limites identifiées
- **Lecture partielle des PDF :** La limite de 20 000 caractères par PDF protège le budget d'API mais tronque la fin des articles longs (souvent les sections *Discussion*, *Limitations* et annexes).
- **Dépendance à la disponibilité d'arXiv :** La recherche automatique cible arXiv pour garantir l'accès libre au fichier PDF complet ; certains articles publiés uniquement chez des éditeurs fermés (IEEE, Springer, Elsevier) ne sont donc pas accessibles par cette voie sans abonnement institutionnel.

#### 3. Principaux apprentissages
- **Maîtrise du *Structured Output* et du *Function Calling* :** Ce projet nous a permis de comprendre concrètement comment transformer un LLM génératif en composant logiciel fiable grâce aux schémas JSON/Pydantic et au mécanisme des `Tool`.
- **Méthodologie de la recherche scientifique :** La structuration d'un état de l'art (tri *keep / borderline / drop*, lecture critique d'objectifs/méthodes/limites) et la construction d'une introduction académique autour d'une lacune de la littérature (*research gap*) constituent d'excellents repères méthodologiques pour nos travaux de recherche en M2 ICE.